# DetectiveQA Baseline 5 — Summary Hybrid Retrieval + LLM Paragraph Rerank

## 1. Install dependencies

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu rank-bm25 pandas numpy tqdm matplotlib google-genai python-dotenv

## 2. Configure local paths

Đặt notebook trong root của repo `lore-router`, hoặc sửa `PROJECT_DIR`.

In [1]:
from pathlib import Path
import os
import json
import math
import re
import time
import random
import hashlib
from collections import defaultdict
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd().parent

RANKING_DIR = PROJECT_DIR / "data" / "detectiveqa-ranking-en"
SUMMARY_DIR = PROJECT_DIR / "data" / "detectiveqa-summary-pass2-en"
OUT_DIR = PROJECT_DIR / "artifacts" / "retrieval_baselines_v5_llm_rerank"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "llm_checkpoints").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "query_checkpoints").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "llm_diagnostics").mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RANKING_DIR:", RANKING_DIR)
print("SUMMARY_DIR:", SUMMARY_DIR)
print("OUT_DIR:", OUT_DIR)


PROJECT_DIR: c:\D\projects\lore-router
RANKING_DIR: c:\D\projects\lore-router\data\detectiveqa-ranking-en
SUMMARY_DIR: c:\D\projects\lore-router\data\detectiveqa-summary-pass2-en
OUT_DIR: c:\D\projects\lore-router\artifacts\retrieval_baselines_v5_llm_rerank


c:\Users\PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Experiment config

In [2]:
MAX_QUERIES = 0          # 0 = all queries; dùng 2-5 để smoke test API
CHUNK_TOP_K = 5          # chunk-summary metric dùng k=5
PARAGRAPH_TOP_N = 100    # candidates trước LLM rerank
FINAL_TOP_K = 10         # final paragraph output

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

CHUNK_DENSE_DEPTH = 100
CHUNK_BM25_DEPTH = 100
CHUNK_DENSE_WEIGHT = 1.0
CHUNK_BM25_WEIGHT = 0.5
RRF_K = 60

PARA_DENSE_ALPHA = 0.85  # paragraph score = alpha*dense + (1-alpha)*bm25
EMBED_BATCH_SIZE = 128

# LLM reranker
LLM_MODEL = "gemma-4-26b-a4b-it"
LLM_TEMPERATURE = 0.0
LLM_SEED = 42
LLM_BATCH_SIZE = 20
LLM_MAX_OUTPUT_TOKENS = 4096
LLM_MAX_RETRIES = 6
LLM_INITIAL_BACKOFF_SECONDS = 2.0
LLM_MAX_BACKOFF_SECONDS = 60.0
LLM_REQUEST_PAUSE_SECONDS = 0.5
MAX_PARAGRAPH_CHARS_FOR_LLM = 1600

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


## 4. Load data

In [3]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_qrels(path: Path):
    qrels = defaultdict(set)
    with path.open("r", encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and float(parts[2]) > 0:
                qrels[parts[0]].add(parts[1])
    return dict(qrels)

corpus_rows = load_jsonl(RANKING_DIR / "corpus.jsonl")
query_rows = load_jsonl(RANKING_DIR / "queries.jsonl")
metadata_rows = load_jsonl(RANKING_DIR / "query_metadata.jsonl")
chunk_summary_rows = load_jsonl(SUMMARY_DIR / "chunk_summaries.jsonl")
paragraph_to_chunk_rows = load_jsonl(SUMMARY_DIR / "paragraph_to_chunk.jsonl")

corpus = {row["_id"]: row for row in corpus_rows}
queries = {row["_id"]: row["text"] for row in query_rows}
query_meta = {row["query_id"]: row for row in metadata_rows}
paragraph_qrels = load_qrels(RANKING_DIR / "qrels" / "test.tsv")

paragraph_to_chunk = {row["paragraph_doc_id"]: row["chunk_id"] for row in paragraph_to_chunk_rows}

chunk_qrels_path = SUMMARY_DIR / "qrels" / "chunks.tsv"
if chunk_qrels_path.exists():
    chunk_qrels = load_qrels(chunk_qrels_path)
else:
    chunk_qrels = {}
    for qid, gold_docs in paragraph_qrels.items():
        chunk_qrels[qid] = {paragraph_to_chunk[d] for d in gold_docs if d in paragraph_to_chunk}

chunk_rows = {row["_id"]: row for row in chunk_summary_rows}

print("Paragraphs:", len(corpus))
print("Queries:", len(queries))
print("Chunk summaries:", len(chunk_rows))
print("Paragraph qrel queries:", len(paragraph_qrels))
print("Chunk qrel queries:", len(chunk_qrels))


Paragraphs: 65641
Queries: 149
Chunk summaries: 986
Paragraph qrel queries: 149
Chunk qrel queries: 149


## 5. Build metadata indexes

In [4]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
DOC_ID_RE = re.compile(r"^dqa-en-(\d+)-p(\d+)$")


def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())


def doc_sort_key(doc_id: str):
    m = DOC_ID_RE.match(doc_id)
    if not m:
        return (10**9, 10**9)
    return (int(m.group(1)), int(m.group(2)))

chunks_by_novel = defaultdict(list)
for row in chunk_summary_rows:
    chunks_by_novel[int(row["novel_id"])].append(row)
for novel_id in chunks_by_novel:
    chunks_by_novel[novel_id].sort(key=lambda r: int(r.get("chunk_index", 0)))

paragraphs_by_novel = defaultdict(list)
for doc_id, row in corpus.items():
    novel_id = int(doc_id.split("-")[2])
    paragraphs_by_novel[novel_id].append(row)
for novel_id in paragraphs_by_novel:
    paragraphs_by_novel[novel_id].sort(key=lambda r: int(r["_id"].split("-p")[-1]))

paragraphs_by_chunk = defaultdict(list)
for row in paragraph_to_chunk_rows:
    paragraphs_by_chunk[row["chunk_id"]].append(row["paragraph_doc_id"])
for chunk_id in paragraphs_by_chunk:
    paragraphs_by_chunk[chunk_id].sort(key=lambda d: int(d.split("-p")[-1]))

query_ids = [qid for qid in queries if qid in paragraph_qrels and qid in query_meta]
if MAX_QUERIES:
    query_ids = query_ids[:MAX_QUERIES]

print("Evaluation queries:", len(query_ids))
print("Novels with chunks:", len(chunks_by_novel))


Evaluation queries: 149
Novels with chunks: 23


## 6. Build dense/BM25 indexes

In [5]:
import torch
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)


Device: cuda


In [6]:
def encode_texts(texts, batch_size=128, normalize=True, show_progress=True):
    return embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
        show_progress_bar=show_progress,
    ).astype("float32")

chunk_index_by_novel = {}
paragraph_index_by_novel = {}

for novel_id in tqdm(sorted(chunks_by_novel), desc="Build chunk indexes"):
    rows = chunks_by_novel[novel_id]
    texts = [r.get("summary", "") for r in rows]
    embeddings = encode_texts(texts, batch_size=EMBED_BATCH_SIZE)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)
    chunk_index_by_novel[novel_id] = {
        "rows": rows,
        "texts": texts,
        "embeddings": embeddings,
        "bm25": bm25,
        "faiss": faiss_index,
    }

for novel_id in tqdm(sorted(paragraphs_by_novel), desc="Build paragraph indexes"):
    rows = paragraphs_by_novel[novel_id]
    texts = [r.get("text", "") for r in rows]
    embeddings = encode_texts(texts, batch_size=EMBED_BATCH_SIZE)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    doc_id_to_pos = {r["_id"]: i for i, r in enumerate(rows)}
    paragraph_index_by_novel[novel_id] = {
        "rows": rows,
        "texts": texts,
        "embeddings": embeddings,
        "bm25": bm25,
        "doc_id_to_pos": doc_id_to_pos,
    }


Build paragraph indexes: 100%|██████████| 23/23 [02:15<00:00,  5.90s/it]


## 7. Metric functions

In [7]:
def precision_at_k(ranked_ids, gold_ids, k):
    top = ranked_ids[:k]
    if not top:
        return 0.0
    return sum(1 for x in top if x in gold_ids) / k


def recall_at_k(ranked_ids, gold_ids, k):
    if not gold_ids:
        return 0.0
    top = ranked_ids[:k]
    return sum(1 for x in top if x in gold_ids) / len(gold_ids)


def reciprocal_rank(ranked_ids, gold_ids):
    for i, x in enumerate(ranked_ids, start=1):
        if x in gold_ids:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranked_ids, gold_ids, k):
    dcg = 0.0
    for i, x in enumerate(ranked_ids[:k], start=1):
        if x in gold_ids:
            dcg += 1.0 / math.log2(i + 1)
    ideal_hits = min(len(gold_ids), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def minmax(values):
    values = np.asarray(values, dtype="float32")
    if len(values) == 0:
        return values
    lo, hi = float(values.min()), float(values.max())
    if hi - lo < 1e-9:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)


## 8. Retrieval functions before LLM rerank

In [8]:
def rrf_rank(dense_order, bm25_order, bm25_scores=None, dense_weight=1.0, bm25_weight=0.5, rrf_k=60):
    scores = defaultdict(float)
    for rank, idx in enumerate(dense_order, start=1):
        scores[int(idx)] += dense_weight / (rrf_k + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        if bm25_scores is not None and bm25_scores[int(idx)] <= 0:
            continue
        scores[int(idx)] += bm25_weight / (rrf_k + rank)
    return [idx for idx, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]


def rank_chunks(query_text, novel_id, top_k=5):
    index = chunk_index_by_novel[novel_id]
    q_emb = encode_texts([QUERY_PREFIX + query_text], batch_size=1, show_progress=False)[0]

    dense_scores = index["embeddings"] @ q_emb
    dense_order = np.argsort(-dense_scores)[:CHUNK_DENSE_DEPTH]

    bm25_scores = np.asarray(index["bm25"].get_scores(tokenize(query_text)), dtype="float32")
    bm25_order = np.argsort(-bm25_scores)[:CHUNK_BM25_DEPTH]

    fused_order = rrf_rank(
        dense_order=dense_order,
        bm25_order=bm25_order,
        bm25_scores=bm25_scores,
        dense_weight=CHUNK_DENSE_WEIGHT,
        bm25_weight=CHUNK_BM25_WEIGHT,
        rrf_k=RRF_K,
    )
    return [index["rows"][i]["_id"] for i in fused_order[:top_k]]


def score_paragraphs(query_text, novel_id, candidate_doc_ids, top_n=100):
    index = paragraph_index_by_novel[novel_id]
    candidate_doc_ids = [d for d in candidate_doc_ids if d in index["doc_id_to_pos"]]
    if not candidate_doc_ids:
        return []

    q_emb = encode_texts([QUERY_PREFIX + query_text], batch_size=1, show_progress=False)[0]
    positions = np.array([index["doc_id_to_pos"][d] for d in candidate_doc_ids], dtype=np.int64)

    dense_scores = index["embeddings"][positions] @ q_emb
    bm25_all = np.asarray(index["bm25"].get_scores(tokenize(query_text)), dtype="float32")
    bm25_scores = bm25_all[positions]

    dense_norm = minmax(dense_scores)
    bm25_norm = minmax(bm25_scores)
    final_scores = PARA_DENSE_ALPHA * dense_norm + (1.0 - PARA_DENSE_ALPHA) * bm25_norm

    order = np.argsort(-final_scores)[:top_n]
    return [(candidate_doc_ids[i], float(final_scores[i])) for i in order]


## 9. Gemini / Gemma API helper for LLM reranking

checkpoint:

- `OUT_DIR/llm_checkpoints`: cache từng batch API.
- `OUT_DIR/query_checkpoints`: cache từng query hoàn chỉnh.

Nếu kernel dừng giữa chừng, chạy lại cell từ phần API trở xuống; các query/batch đã xong sẽ được reuse.


In [9]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

RERANK_PROMPT_VERSION = "detectiveqa-llm-rerank-v1.0"

LLM_SYSTEM_INSTRUCTION = """You are a careful evidence reranker for detective-fiction question answering.
Your job is to score candidate paragraphs by how useful they are as evidence for answering the query.
Do not output hidden reasoning. Return only the requested JSON array."""


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_api_key():
    load_dotenv(PROJECT_DIR / ".env")
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not key:
        raise RuntimeError("No GEMINI_API_KEY or GOOGLE_API_KEY found in .env")
    return key


def extract_visible_response_text(response: Any) -> str:
    fragments = []
    for candidate in getattr(response, "candidates", None) or []:
        content = getattr(candidate, "content", None)
        for part in getattr(content, "parts", None) or []:
            if bool(getattr(part, "thought", False)):
                continue
            text = getattr(part, "text", None)
            if isinstance(text, str) and text.strip():
                fragments.append(text.strip())
    if fragments:
        return "\n".join(fragments).strip()
    try:
        text = getattr(response, "text", None)
    except Exception:
        text = None
    return text.strip() if isinstance(text, str) else ""


def exception_status_code(exc: Exception):
    for attribute in ("code", "status_code"):
        value = getattr(exc, attribute, None)
        if isinstance(value, int):
            return value
        try:
            if value is not None:
                return int(value)
        except (TypeError, ValueError):
            pass
    match = re.search(r"\b(429|500|502|503|504)\b", str(exc))
    return int(match.group(1)) if match else None


api_key = load_api_key()
llm_client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=180_000),
)
print("Gemini client initialized. API key is not displayed.")


Gemini client initialized. API key is not displayed.


In [10]:

def truncate_for_llm(text: str, max_chars: int = MAX_PARAGRAPH_CHARS_FOR_LLM) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."


def build_llm_rerank_prompt(query_text: str, candidates: list[tuple[str, float]]) -> str:
    items = []
    for rank, (doc_id, base_score) in enumerate(candidates, start=1):
        text = truncate_for_llm(corpus[doc_id]["text"])
        items.append(
            f"Candidate {rank}\n"
            f"doc_id: {doc_id}\n"
            f"baseline_score: {base_score:.6f}\n"
            f"text: {text}"
        )
    candidate_block = "\n\n".join(items)
    return f"""Rank candidate paragraphs for a DetectiveQA evidence retrieval benchmark.

Query:
{query_text}

Scoring rubric:
- 3 = strong evidence/clue needed to answer the query.
- 2 = useful partial evidence.
- 1 = weakly related, mostly surface word overlap, later restatement, or answer-like mention without useful supporting evidence.
- 0 = irrelevant.

Important:
- Prefer paragraphs that contain evidence, clues, observations, facts, motives, or constraints needed to answer the query.
- Do not give high scores merely because a paragraph repeats words from the query.
- Do not automatically prefer later testimony or answer-like restatements if they do not contain the underlying supporting evidence.
- Score each paragraph independently based only on the query and the paragraph text.
- Use the provided doc_id values exactly.

Return only a JSON array. Each item must have exactly these fields:
{{"doc_id": "...", "score": 0}}

Candidate paragraphs:
{candidate_block}
"""


def strip_markdown_fences(text: str) -> str:
    raw = str(text).strip()
    # Handles ```json ... ``` and ``` ... ``` even when there is text around it.
    raw = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```\s*$", "", raw)
    return raw.strip()


def extract_first_json_payload(text: str):
    """Extract the first complete JSON array/object from a messy model response.

    The previous parser used a greedy regex like \[.*\]. If the model returned
    two arrays or added extra text after the array, json.loads crashed with
    'Extra data'. This decoder-based parser accepts the first complete JSON
    payload and ignores trailing text.
    """
    raw = strip_markdown_fences(text)
    decoder = json.JSONDecoder()

    # Fast path: full response is clean JSON.
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Prefer arrays, because the prompt asks for an array. Try every '['.
    for match in re.finditer(r"\[", raw):
        try:
            payload, _end = decoder.raw_decode(raw[match.start():])
            if isinstance(payload, list):
                return payload
        except json.JSONDecodeError:
            continue

    # Fallback: some models wrap results in {"scores": [...]}.
    for match in re.finditer(r"\{", raw):
        try:
            payload, _end = decoder.raw_decode(raw[match.start():])
            if isinstance(payload, dict):
                return payload
        except json.JSONDecodeError:
            continue

    raise ValueError("No complete JSON array/object found in LLM response")


def parse_llm_scores(text: str, valid_doc_ids: set[str]) -> dict[str, float]:
    payload = extract_first_json_payload(text)

    if isinstance(payload, dict):
        # Accept common wrappers without requiring a second API call.
        for key in ("scores", "results", "items", "rankings"):
            if isinstance(payload.get(key), list):
                payload = payload[key]
                break

    if not isinstance(payload, list):
        raise ValueError("LLM response is not a JSON array or supported wrapper object")

    scores = {}
    for item in payload:
        if not isinstance(item, dict):
            continue
        doc_id = str(item.get("doc_id", "")).strip()
        if doc_id not in valid_doc_ids:
            continue
        try:
            score = float(item.get("score", 0))
        except (TypeError, ValueError):
            score = 0.0
        scores[doc_id] = max(0.0, min(3.0, score))
    return scores


def generation_request_sha256(prompt: str, max_output_tokens: int) -> str:
    request = {
        "prompt_version": RERANK_PROMPT_VERSION,
        "system_instruction": LLM_SYSTEM_INSTRUCTION,
        "prompt": prompt,
        "model": LLM_MODEL,
        "temperature": LLM_TEMPERATURE,
        "seed": LLM_SEED,
        "max_output_tokens": max_output_tokens,
    }
    return sha256_text(json.dumps(request, ensure_ascii=False, sort_keys=True))


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    tmp_path.replace(path)


def call_llm_with_retry(prompt: str, max_output_tokens: int = LLM_MAX_OUTPUT_TOKENS) -> str:
    retryable_statuses = {429, 500, 502, 503, 504}
    for attempt in range(LLM_MAX_RETRIES + 1):
        try:
            response = llm_client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=LLM_SYSTEM_INSTRUCTION,
                    temperature=LLM_TEMPERATURE,
                    seed=LLM_SEED,
                    max_output_tokens=max_output_tokens,
                    candidate_count=1,
                    response_mime_type="text/plain",
                    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                    tool_config=types.ToolConfig(
                        function_calling_config=types.FunctionCallingConfig(mode="NONE")
                    ),
                    thinking_config=types.ThinkingConfig(
                        thinking_level=types.ThinkingLevel.MINIMAL,
                        include_thoughts=False,
                    ),
                ),
            )
            text = extract_visible_response_text(response)
            if text:
                if LLM_REQUEST_PAUSE_SECONDS > 0:
                    time.sleep(LLM_REQUEST_PAUSE_SECONDS)
                return text
            raise RuntimeError("API returned no visible text")
        except Exception as exc:
            status = exception_status_code(exc)
            retryable = status in retryable_statuses or status is None
            if attempt >= LLM_MAX_RETRIES or not retryable:
                raise
            backoff = min(LLM_MAX_BACKOFF_SECONDS, LLM_INITIAL_BACKOFF_SECONDS * (2 ** attempt))
            jitter = random.uniform(0.0, min(1.0, backoff * 0.1))
            wait_seconds = backoff + jitter
            print(f"LLM API failed status={status}, attempt={attempt+1}/{LLM_MAX_RETRIES+1}: {exc}. Retrying in {wait_seconds:.1f}s")
            time.sleep(wait_seconds)
    raise AssertionError("Unreachable retry loop")


def batch_checkpoint_path(qid: str, batch_index: int, request_hash: str) -> Path:
    safe_qid = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(qid))
    return OUT_DIR / "llm_checkpoints" / f"{safe_qid}_b{batch_index:03d}_{request_hash[:12]}.json"


def llm_score_batch(qid: str, batch_index: int, query_text: str, candidates: list[tuple[str, float]]):
    prompt = build_llm_rerank_prompt(query_text, candidates)
    request_hash = generation_request_sha256(prompt, LLM_MAX_OUTPUT_TOKENS)
    checkpoint_path = batch_checkpoint_path(qid, batch_index, request_hash)
    valid_doc_ids = {doc_id for doc_id, _ in candidates}

    if checkpoint_path.exists():
        payload = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        if "scores" in payload:
            return {k: float(v) for k, v in payload["scores"].items()}
        # Backward-compatible recovery from a raw-only diagnostic checkpoint.
        if "raw_response" in payload:
            try:
                scores = parse_llm_scores(payload["raw_response"], valid_doc_ids)
            except Exception:
                scores = {}
            for doc_id, _ in candidates:
                scores.setdefault(doc_id, 0.0)
            payload["scores"] = scores
            payload["parse_ok"] = bool(scores)
            atomic_write_json(checkpoint_path, payload)
            return scores

    raw_response = call_llm_with_retry(prompt, max_output_tokens=LLM_MAX_OUTPUT_TOKENS)

    parse_ok = True
    parse_error = None
    try:
        scores = parse_llm_scores(raw_response, valid_doc_ids)
    except Exception as exc:
        # Never stop the whole benchmark because one LLM response was malformed.
        # All-zero LLM scores preserve the original paragraph hybrid order via tie-breaker.
        parse_ok = False
        parse_error = repr(exc)
        scores = {}
        diagnostic_path = OUT_DIR / "llm_diagnostics" / f"parse_error_{qid}_b{batch_index:03d}_{request_hash[:12]}.json"
        atomic_write_json(diagnostic_path, {
            "query_id": qid,
            "batch_index": batch_index,
            "request_sha256": request_hash,
            "parse_error": parse_error,
            "raw_response": raw_response,
            "doc_ids": [doc_id for doc_id, _ in candidates],
        })
        print(f"Warning: could not parse LLM JSON for query={qid} batch={batch_index}. Falling back to base order. Diagnostic: {diagnostic_path}")

    # Missing candidates get score 0 rather than failing the whole run.
    for doc_id, _ in candidates:
        scores.setdefault(doc_id, 0.0)

    payload = {
        "query_id": qid,
        "batch_index": batch_index,
        "request_sha256": request_hash,
        "model": LLM_MODEL,
        "temperature": LLM_TEMPERATURE,
        "seed": LLM_SEED,
        "prompt_version": RERANK_PROMPT_VERSION,
        "doc_ids": [doc_id for doc_id, _ in candidates],
        "scores": scores,
        "parse_ok": parse_ok,
        "parse_error": parse_error,
        "raw_response": raw_response,
    }
    atomic_write_json(checkpoint_path, payload)
    return scores


def llm_rerank_paragraphs(qid: str, query_text: str, scored_doc_ids: list[tuple[str, float]]):
    all_scores = {}
    for batch_index, start in enumerate(range(0, len(scored_doc_ids), LLM_BATCH_SIZE)):
        batch = scored_doc_ids[start:start + LLM_BATCH_SIZE]
        batch_scores = llm_score_batch(qid, batch_index, query_text, batch)
        all_scores.update(batch_scores)

    # Final order: LLM score desc, then original hybrid score desc, then original order.
    ranked = []
    for original_rank, (doc_id, base_score) in enumerate(scored_doc_ids, start=1):
        ranked.append({
            "doc_id": doc_id,
            "llm_score": float(all_scores.get(doc_id, 0.0)),
            "base_score": float(base_score),
            "original_rank": original_rank,
        })
    ranked.sort(key=lambda r: (r["llm_score"], r["base_score"], -r["original_rank"]), reverse=True)
    return ranked


## 10. Run baseline 5

In [11]:

def query_checkpoint_path(qid: str) -> Path:
    safe_qid = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(qid))
    return OUT_DIR / "query_checkpoints" / f"{safe_qid}.json"


def load_query_checkpoint(qid: str) -> dict[str, Any] | None:
    path = query_checkpoint_path(qid)
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("prompt_version") != RERANK_PROMPT_VERSION:
        return None
    if payload.get("llm_model") != LLM_MODEL:
        return None
    if int(payload.get("paragraph_top_n", -1)) != int(PARAGRAPH_TOP_N):
        return None
    if int(payload.get("chunk_top_k", -1)) != int(CHUNK_TOP_K):
        return None
    return payload


def evaluate_one_query(qid: str) -> dict[str, Any]:
    query_text = queries[qid]
    novel_id = int(query_meta[qid]["novel_id"])

    selected_chunks = rank_chunks(query_text, novel_id, top_k=CHUNK_TOP_K)
    gold_chunks = chunk_qrels.get(qid, set())
    gold_paragraphs = paragraph_qrels.get(qid, set())

    chunk_metric_row = {
        "query_id": qid,
        "novel_id": novel_id,
        "chunk_precision_at_5": precision_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "chunk_recall_at_5": recall_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "chunk_mrr": reciprocal_rank(selected_chunks, gold_chunks),
        "chunk_ndcg_at_5": ndcg_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "gold_chunk_count": len(gold_chunks),
    }

    candidate_doc_ids = []
    seen = set()
    for chunk_id in selected_chunks:
        for doc_id in paragraphs_by_chunk.get(chunk_id, []):
            if doc_id not in seen:
                candidate_doc_ids.append(doc_id)
                seen.add(doc_id)

    scored_top_n = score_paragraphs(query_text, novel_id, candidate_doc_ids, top_n=PARAGRAPH_TOP_N)
    top_n_ids = [doc_id for doc_id, _ in scored_top_n]

    llm_ranked_rows = llm_rerank_paragraphs(qid, query_text, scored_top_n)
    final_ranking = [row["doc_id"] for row in llm_ranked_rows[:FINAL_TOP_K]]

    paragraph_metric_row = {
        "query_id": qid,
        "novel_id": novel_id,
        "precision_at_10": precision_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "recall_at_10": recall_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "mrr": reciprocal_rank(final_ranking, gold_paragraphs),
        "ndcg_at_10": ndcg_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "candidate_recall_at_100": recall_at_k(top_n_ids, gold_paragraphs, PARAGRAPH_TOP_N),
        "reachable_recall_from_chunks": recall_at_k(candidate_doc_ids, gold_paragraphs, len(candidate_doc_ids)),
        "gold_paragraph_count": len(gold_paragraphs),
        "selected_chunk_count": len(selected_chunks),
        "candidate_paragraph_count": len(candidate_doc_ids),
        "paragraph_top_n": len(scored_top_n),
    }

    run_chunk_row = {
        "query_id": qid,
        "novel_id": novel_id,
        "query": query_text,
        "ranked_chunk_ids": selected_chunks,
        "gold_chunk_ids": sorted(gold_chunks),
    }

    run_paragraph_row = {
        "query_id": qid,
        "novel_id": novel_id,
        "query": query_text,
        "selected_chunk_ids": selected_chunks,
        "candidate_paragraph_count": len(candidate_doc_ids),
        "top_before_llm_rerank": [
            {"doc_id": doc_id, "base_score": base_score}
            for doc_id, base_score in scored_top_n
        ],
        "top_after_llm_rerank": llm_ranked_rows[:FINAL_TOP_K],
        "gold_doc_ids": sorted(gold_paragraphs, key=doc_sort_key),
    }

    return {
        "query_id": qid,
        "prompt_version": RERANK_PROMPT_VERSION,
        "llm_model": LLM_MODEL,
        "chunk_top_k": CHUNK_TOP_K,
        "paragraph_top_n": PARAGRAPH_TOP_N,
        "final_top_k": FINAL_TOP_K,
        "paragraph_metric_row": paragraph_metric_row,
        "chunk_metric_row": chunk_metric_row,
        "run_paragraph_row": run_paragraph_row,
        "run_chunk_row": run_chunk_row,
    }


paragraph_metric_rows = []
chunk_metric_rows = []
run_paragraph_rows = []
run_chunk_rows = []

for qid in tqdm(query_ids, desc="Evaluate with LLM rerank"):
    cached = load_query_checkpoint(qid)
    if cached is not None:
        result = cached
    else:
        result = evaluate_one_query(qid)
        atomic_write_json(query_checkpoint_path(qid), result)

    paragraph_metric_rows.append(result["paragraph_metric_row"])
    chunk_metric_rows.append(result["chunk_metric_row"])
    run_paragraph_rows.append(result["run_paragraph_row"])
    run_chunk_rows.append(result["run_chunk_row"])

paragraph_metrics = pd.DataFrame(paragraph_metric_rows)
chunk_metrics = pd.DataFrame(chunk_metric_rows)

paragraph_summary = paragraph_metrics[[
    "precision_at_10", "recall_at_10", "mrr", "ndcg_at_10",
    "candidate_recall_at_100", "reachable_recall_from_chunks",
    "candidate_paragraph_count", "paragraph_top_n",
]].mean().to_dict()

chunk_summary = chunk_metrics[[
    "chunk_precision_at_5", "chunk_recall_at_5", "chunk_mrr", "chunk_ndcg_at_5",
]].mean().to_dict()

summary_rows = [
    {
        "system": "summary_hybrid_chunks5_para100_llm_rerank10",
        "level": "paragraph",
        **paragraph_summary,
    },
    {
        "system": "summary_hybrid_chunks5",
        "level": "chunk_summary",
        **chunk_summary,
    },
]
summary_df = pd.DataFrame(summary_rows)
summary_df


Evaluate with LLM rerank:  30%|███       | 45/149 [03:12<1:26:31, 49.91s/it]

LLM API failed status=500, attempt=1/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 2.1s


Evaluate with LLM rerank:  31%|███       | 46/149 [06:09<2:30:52, 87.89s/it]

LLM API failed status=500, attempt=1/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 2.0s
LLM API failed status=500, attempt=2/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 4.1s


Evaluate with LLM rerank:  32%|███▏      | 47/149 [08:40<3:01:44, 106.91s/it]

...

LLM API failed status=500, attempt=1/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 2.0s
LLM API failed status=500, attempt=2/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 4.1s
LLM API failed status=500, attempt=3/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 8.1s
LLM API failed status=500, attempt=1/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 2.0s


Evaluate with LLM rerank:  97%|█████████▋| 145/149 [3:17:34<08:37, 129.48s/it]

LLM API failed status=500, attempt=1/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 2.0s
LLM API failed status=500, attempt=2/7: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. Retrying in 4.1s


Evaluate with LLM rerank: 100%|██████████| 149/149 [3:24:24<00:00, 82.31s/it] 


,system,level,precision_at_10,recall_at_10,mrr,ndcg_at_10,candidate_recall_at_100,reachable_recall_from_chunks,candidate_paragraph_count,paragraph_top_n,chunk_precision_at_5,chunk_recall_at_5,chunk_mrr,chunk_ndcg_at_5
0,summary_hybrid_chunks5_para100_llm_rerank10,paragraph,0.049664,0.11291,0.192306,0.103971,0.243576,0.37185,326.805369,100.0,NaN,NaN,NaN,NaN
1,summary_hybrid_chunks5,chunk_summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.189262,0.359641,0.401566,0.317975


In [13]:
summary_df.drop(columns=['system'])

,level,precision_at_10,recall_at_10,mrr,ndcg_at_10,candidate_recall_at_100,reachable_recall_from_chunks,candidate_paragraph_count,paragraph_top_n,chunk_precision_at_5,chunk_recall_at_5,chunk_mrr,chunk_ndcg_at_5
0,paragraph,0.049664,0.11291,0.192306,0.103971,0.243576,0.37185,326.805369,100.0,NaN,NaN,NaN,NaN
1,chunk_summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.189262,0.359641,0.401566,0.317975


## 13. Inspect one query

In [14]:
idx = 0
row = run_paragraph_rows[idx]
print("Query:", row["query"])
print("Novel:", row["novel_id"])
print("Gold docs:", row["gold_doc_ids"])
print("Selected chunks:", row["selected_chunk_ids"])
print("Candidate paragraphs:", row["candidate_paragraph_count"])
print("\nTop 10 after LLM rerank:")
for rank, item in enumerate(row["top_after_llm_rerank"], start=1):
    doc_id = item["doc_id"]
    text = corpus[doc_id]["text"]
    print(f"{rank:02d}  {doc_id}  llm={item['llm_score']}  base={item['base_score']:.4f}")
    print(text[:500].replace("\n", " "))
    print()


Query: Alyona advised Polo not to tell anyone when she left, which implies that:
Novel: 117
Gold docs: ['dqa-en-117-p92', 'dqa-en-117-p94', 'dqa-en-117-p98', 'dqa-en-117-p101', 'dqa-en-117-p103', 'dqa-en-117-p107', 'dqa-en-117-p226', 'dqa-en-117-p229', 'dqa-en-117-p246', 'dqa-en-117-p441', 'dqa-en-117-p478']
Selected chunks: ['dqa-en-117-c0019', 'dqa-en-117-c0014', 'dqa-en-117-c0012', 'dqa-en-117-c0021', 'dqa-en-117-c0003']
Candidate paragraphs: 360

Top 10 after LLM rerank:
01  dqa-en-117-p883  llm=3.0  base=1.0000
Polo said, "Yes, because Mrs. Marshall asked me not to tell anyone I saw her this morning when she left the seaside. I immediately realized that there was trouble between her and her husband due to her relationship with Patrick Redfern. I thought she had a rendezvous with Patrick Redfern somewhere, but I hoped to avoid her husband's eyes."

02  dqa-en-117-p906  llm=2.0  base=0.4418
"She said she'd meet me in the hall at ten-thirty, and I was afraid I'd be late, so I wasn't.